In [27]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as mpl
import seaborn as sb
import missingno as msn
import warnings
warnings.filterwarnings('ignore')

print('OK')

OK


In [28]:
train = pd.read_parquet('C:/Users/Deepayan/Documents/MAVERICK/projects/churn-prediction/artifacts/kkbox_final.parquet')
print('OK')

OK


In [29]:
train.shape, train.dtypes, train.memory_usage(deep=True).sum() / 1e6

((970960, 19),
 msno                          str
 is_churn                    int64
 city                      float64
 bd                        float64
 gender                        str
 registered_via            float64
 registration_init_time    float64
 transanction_count        float64
 first_transaction_date    float64
 last_transaction_date     float64
 last_plan_days            float64
 last_plan_price           float64
 avg_amount_paid           float64
 auto_renew_flag           float64
 cancel_count              float64
 total_secs                float64
 num_unq                   float64
 num_100                   float64
 days_active               float64
 dtype: object,
 np.float64(192.35397))

In [30]:
train['msno'].duplicated().sum()

np.int64(0)

In [31]:
train['city'].isnull().sum() #pandas treats NaN values by upcasting them to float64, so check for null values

np.int64(109993)

In [32]:
train['registered_via'].isnull().sum() #pandas treats NaN values by upcasting them to float64, so check for null values

np.int64(109993)

In [33]:
whether_city_registered_via_have_same_missing_rows = train.loc[train['city'].isnull(), 'msno'].equals(train.loc[train['registered_via'].isnull(), 'msno'])
print(whether_city_registered_via_have_same_missing_rows) 

#checking is the same rows have both city and registered_via missing
#if that is true, it confirms this is not a merge bug

True


In [34]:
train['has_details'] = train['city'].notnull()
pd.crosstab(train['has_details'], train['is_churn'], normalize= 'index') 

#relates the is_churn column with 'city' and 'registered_via' when they are null and not null

is_churn,0,1
has_details,,
False,0.946524,0.053476
True,0.905399,0.094601


-> Users who have signed up and completed their profile (filled in their 'city' and 'registered_via' info in details) have churned at 5.3%
-> Users who have signed up but not completed their profile (filled in their 'city' and 'registered_via' info in details) have churned at 9.5%

Hence, profile incompleteness does not directly lead to user being churned as nearly double the users have actually churned who have complete profiles than those who have incomplete profiles. 

In [35]:
train.groupby('has_details')['registration_init_time'].describe()

,count,mean,std,min,25%,50%,75%,max
has_details,,,,,,,,
False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
True,860967.0,2.013265e+07,30111.744263,20040326.0,20120214.0,20140602.0,20160118.0,20170424.0


This confirms that all users who have 'city' and 'registered_via' details missing also have 'registration_init_time' blank, strongly leading to the conclusion that these users ('msno') do not exist in the members.csv file but since they exist in the merged parquet, they must have records in transactions.csv and user_logs.csv

In [36]:
train.groupby('has_details')['first_transaction_date'].describe()

,count,mean,std,min,25%,50%,75%,max
has_details,,,,,,,,
False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
True,1.0,20151012.0,NaN,20151012.0,20151012.0,20151012.0,20151012.0,20151012.0


COUNT

Out of 109993 users with no details, 108210 users have a transaction date in the window which means there are 1783 users with no details who also have zero transactions in this window.

->The mean, median, 25, 75 percentiles all have dates having a gap of nearly 1-2 days for whether the person has details or not. This effectively rules out the assumption that there must have been a legacy account which logged in users during earlier times without details who have somehow managed to not churn till now before introducing a modified subscription plan for new users requiring their details who churn at a faster rate.

MEDIAN (50%)

The min column shows that earliest transactions date back to 2015-01-02 but the median column shows 2017-03-16, much closer to the max column showing 2017-03-31. This effectively means that there have been significantly more transactions in the last 2-3 weeks of a 2 year window.

In [37]:
print(train['first_transaction_date'].min(), train['last_transaction_date'].max())

train['has_transactions'] = train['transanction_count'].notnull()
pd.crosstab(train['has_transactions'], train['is_churn'], normalize = 'index')


20151012.0 20151215.0


is_churn,0,1
has_transactions,,
False,0.910059,0.089941
True,0.000000,1.000000


INITIALLY

The full range of the recorded transactions in the dataset is from 1st January, 2015 to 31st March, 2017.

Crosstabing whether a user has transactions with their respective churn rates highlights two thing:

1. 78.7% of users who don't have any transactions in this window have churned.
2. Only 6.2% of users who have actually recorded transactions have churned.

NOW

After modifications in feature_engineering.py to not consider transactions in my assumed membership expiration month (February 2017), the new churn data shows a strong inclination that my assumption must have been wron (the exipiration month must be something else).

Checking whether churned vs non-churned users' last transaction dates diverge in a way consistent with label leakage from the renewal-check window

In [38]:
train['last_transaction_date_dt'] = pd.to_datetime(train['last_transaction_date'], format= '%Y%m%d')
train.groupby('is_churn')['last_transaction_date_dt'].describe()

,count,mean,min,25%,50%,75%,max
is_churn,,,,,,,
0,0,NaT,NaT,NaT,NaT,NaT,NaT
1,1,2015-12-15 00:00:00,2015-12-15 00:00:00,2015-12-15 00:00:00,2015-12-15 00:00:00,2015-12-15 00:00:00,2015-12-15 00:00:00


INITIALLY

1. The train dataset description lists February 2017 as the month when user subscriptions expire. A 30-day renewal period goes into March 2017.

2. From the description, the median and 75th percentiles show that most of the users who have at least one transaction have their last transaction date, roughly the same without much difference for churned and non-churned user (only a 5-day gap).

So the earlier narrative that no transactions at all = churn provides superficial information only. So the criteria should not be 'has_transactions', instead it should be 'has_transactions_after_feb_2017'.

NOW

Only 1 row in the count columns suggests that either the merge in feature_engineering.py must have been faulty or the expiration month is some other month, not February 2017.

In [41]:
transactions_raw = pd.read_csv("C:/Users/Deepayan/Documents/MAVERICK/projects/churn-prediction/data/raw/kkbox/transactions_v2.csv")
print(transactions_raw.shape)
print(transactions_raw['membership_expire_date'].dtype)

feb_mask = (transactions_raw['membership_expire_date'] >= 20170201) & \
           (transactions_raw['membership_expire_date'] <= 20170228)
print(feb_mask.sum(), transactions_raw.loc[feb_mask, 'msno'].nunique())

(1431009, 9)
int64
350 350


The above data shows that there are only 350 rows which suggest expiration in the month of February 2017 out of 1.4 million transaction records. So the expiration month is confirmed to be not February 2017.

In [42]:
train_msno = train['msno']

common_msno = transactions_raw[transactions_raw['msno'].isin(train_msno)]

common_msno['membership_expire_date'].astype(str).str[:6].value_counts().sort_index()

membership_expire_date
201702        25
201703     43032
201704    906210
201705     94482
201706     15123
           ...  
202304         3
202305         3
202306         1
202307         1
202308         1
Name: count, Length: 79, dtype: int64